# Foundry Weather Agent

This notebook creates a Microsoft Foundry weather agent that uses Open-Meteo for live location weather and provides clothing, activity, and safety recommendations. Authentication uses `DefaultAzureCredential` and the existing `az login` session.

In [18]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

# Find the .env file in this folder or one of its parents.
def find_env_file() -> Path:
    for directory in [Path.cwd(), *Path.cwd().parents]:
        candidate = directory / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find .env in the current directory or a parent directory.")

env_file = find_env_file()
load_dotenv(env_file)
foundry_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
model_name = os.environ["MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
foundry_client = AIProjectClient(endpoint=foundry_endpoint, credential=credential)
chat_client = foundry_client.get_openai_client()
print(f"Loaded {env_file}")
print(f"Using model deployment: {model_name}")

Loaded c:\Azure AI\AI_For_The_Azure_Practice_Entry_Level\Code\WeatherAgent\.env
Using model deployment: gpt-5-mini


In [20]:
geocoding_openapi_spec = {
    "openapi": "3.1.0",
    "info": {
        "title": "Open-Meteo Geocoding API",
        "version": "1.0.0",
        "description": "Find coordinates for a city or place.",
    },
    "servers": [{"url": "https://geocoding-api.open-meteo.com"}],
    "paths": {
        "/v1/search": {
            "get": {
                "operationId": "geocode_location",
                "summary": "Find coordinates for a city or place",
                "parameters": [
                    {"name": "name", "in": "query", "required": True, "schema": {"type": "string"}},
                    {"name": "count", "in": "query", "schema": {"type": "integer", "default": 1, "maximum": 5}},
                    {"name": "language", "in": "query", "schema": {"type": "string", "default": "en"}},
                    {"name": "format", "in": "query", "schema": {"type": "string", "default": "json"}},
                ],
                "responses": {
                    "200": {
                        "description": "Matching locations",
                        "content": {"application/json": {"schema": {"type": "object"}}},
                    }
                },
            }
        }
    },
}

forecast_openapi_spec = {
    "openapi": "3.1.0",
    "info": {
        "title": "Open-Meteo Forecast API",
        "version": "1.0.0",
        "description": "Retrieve current and forecast weather.",
    },
    "servers": [{"url": "https://api.open-meteo.com"}],
    "paths": {
        "/v1/forecast": {
            "get": {
                "operationId": "get_weather_forecast",
                "summary": "Get current and forecast weather",
                "parameters": [
                    {"name": "latitude", "in": "query", "required": True, "schema": {"type": "number"}},
                    {"name": "longitude", "in": "query", "required": True, "schema": {"type": "number"}},
                    {"name": "current", "in": "query", "schema": {"type": "string", "default": "temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,showers,snowfall,weather_code,cloud_cover,wind_speed_10m,wind_gusts_10m"}},
                    {"name": "hourly", "in": "query", "schema": {"type": "string", "default": "temperature_2m,apparent_temperature,precipitation_probability,weather_code,wind_speed_10m"}},
                    {"name": "forecast_days", "in": "query", "schema": {"type": "integer", "default": 3, "minimum": 1, "maximum": 7}},
                    {"name": "timezone", "in": "query", "schema": {"type": "string", "default": "auto"}},
                    {"name": "temperature_unit", "in": "query", "schema": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"}},
                ],
                "responses": {
                    "200": {
                        "description": "Weather observations and forecast",
                        "content": {"application/json": {"schema": {"type": "object"}}},
                    }
                },
            }
        }
    },
}

def openapi_tool(name: str, spec: dict) -> dict:
    return {
        "type": "openapi",
        "openapi": {
            "name": name,
            "spec": spec,
            "auth": {"type": "anonymous"},
        },
    }

geocoding_tool = openapi_tool("open_meteo_geocoding", geocoding_openapi_spec)
forecast_tool = openapi_tool("open_meteo_forecast", forecast_openapi_spec)
print("Weather API tools configured.")

Weather API tools configured.


In [21]:
weather_agent_name = "weather-activity-advisor"
weather_instructions = """
You are a practical weather and activity advisor.

For every location-specific request, use the Open-Meteo tools to geocode the location and retrieve current or forecast weather before answering. Never invent weather values. State the location, local time or forecast period, and the relevant retrieved conditions.

Give concise, practical recommendations for clothing, outdoor or indoor activities, and safety. Consider temperature, feels-like temperature, rain probability, precipitation, snow, wind, and daylight. Recommend layers, waterproof clothing, sun protection, hydration, or indoor alternatives when appropriate. For dangerous or severe conditions, advise checking official local alerts. Ask a clarifying question if the location or time period is missing or ambiguous.
"""

weather_agent = foundry_client.agents.create_version(
    agent_name=weather_agent_name,
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=weather_instructions,
        tools=[geocoding_tool, forecast_tool],
    ),
)
print(f"Weather agent ready: {weather_agent.name} (version {weather_agent.version})")

Weather agent ready: weather-activity-advisor (version 4)


In [22]:
chat_session = chat_client.conversations.create()


def ask_weather_agent(question: str) -> str:
    result = chat_client.responses.create(
        conversation=chat_session.id,
        extra_body={
            "agent": {
                "name": weather_agent_name,
                "type": "agent_reference",
            }
        },
        input=question,
    )
    return result.output_text

print(ask_weather_agent("What should I wear and do outside in Stockholm today evening?"))

Location: Stockholm, Sweden (Europe/Stockholm, local time UTC+2).
Forecast period used: today evening (approx. 18:00–22:00 local).

Retrieved conditions (from hourly forecast)
- 18:00 — temp 20.2 °C (feels 19.7 °C), precipitation probability 12%, wind ~4.7 km/h (gusts ~14 km/h), mostly cloudy.
- 19:00 — temp 19.6 °C (feels 19.0 °C), precip prob 5%, light wind.
- 20:00–22:00 — temps falling from ~18.7 °C to ~16.2 °C (feels similar), precip prob ~0%, light winds, partly cloudy. No measurable precipitation expected.

What to wear
- Base: t‑shirt or light long sleeve.
- Layer: thin sweater or light jacket (temperatures drop a few degrees after sunset).
- Optional: packable light rain jacket in your bag if you want extra peace of mind (low chance of shower early evening but mostly dry).
- Footwear: comfortable shoes; if you’ll be near water or on bikes, shoes with good grip.
- If you’ll be out later at night, add a warmer layer (temperatures near 15–16 °C) and consider reflective clothing o

## Test prompts

Run additional questions with `ask_weather_agent(...)`:

- `What should I wear in London tomorrow morning?`
- `Is this afternoon suitable for cycling in Amsterdam?`
- `Suggest indoor activities in New York if it rains tomorrow.`
- `What is the weather in Springfield?` to test ambiguous locations.

Weather information is advisory and should not replace official emergency or severe-weather alerts.